# Day 27 — Observability: logging & tracing

The eval harness (Day 26) tells you about a frozen set. **Production** needs observability: a
user reports a bad answer, and you need to see every stage's input and output *after the
fact*. We build a tracer from scratch, instrument a RAG pipeline, render the trace tree, and
walk the "find the bug" workflow — then map it onto LangSmith / Langfuse / OpenTelemetry.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | The three pillars: logs, traces, metrics | 4 min |
| 1 | A trace is nested spans — build a tracer | 14 min |
| 2 | Instrument a RAG pipeline | 12 min |
| 3 | What to capture at each stage | 10 min |
| 4 | Debugging from a trace | 10 min |
| 5 | Redaction, sampling, cost; the real tools | 7 min |
| 6 | Exercises + quiz | 3 min |

Kernel: **Python (ai-upskill)**.

In [1]:
import time, json, uuid, contextvars, hashlib
from contextlib import contextmanager
import numpy as np
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer("all-MiniLM-L6-v2")
def E(x): return emb.encode(x if isinstance(x, list) else [x], normalize_embeddings=True)
print("ready")

/Users/umeshkaranam/Desktop/personal/UPSKILL/AI/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7996.99it/s]

ready


## 0 — The three pillars (4 min)

| Pillar | Question | Shape | Example |
| ------ | -------- | ----- | ------- |
| **Logs** | what happened at this moment? | timestamped structured events | `{"evt":"retrieval","query":"...","n_hits":3,"top_score":0.71}` |
| **Traces** | what was the full path of *this one request*? | a tree of timed spans | retrieve → rerank → generate → parse |
| **Metrics** | how is the system doing *in aggregate*? | numbers over time | p95 latency, error rate, cost/day, thumbs-down rate |

For LLM apps the **trace** is the star: one request fans out into retrieval, reranking, tool
calls, and generation, and a bad final answer could come from any of them. Without the trace
you're guessing.

## 1 — Build a tracer (14 min)

A **span** is one timed operation: `name`, `start`/`end`, `inputs`, `outputs`, `metadata`,
`status`, and a `parent`. A **trace** is the root span plus its descendants. We use a
`contextvar` to track the current parent so nested `with span(...)` calls attach correctly.

In [2]:
_current = contextvars.ContextVar("span", default=None)

class Span:
    def __init__(self, name, kind="op"):
        self.id = uuid.uuid4().hex[:8]
        self.name, self.kind = name, kind
        self.parent = _current.get()
        self.children = []
        self.inputs = {}; self.outputs = {}; self.metadata = {}
        self.status = "ok"; self.error = None
        self.start = self.end = None
        if self.parent: self.parent.children.append(self)
    @property
    def duration_ms(self):
        return None if self.end is None else (self.end - self.start) * 1000

@contextmanager
def span(name, kind="op", **inputs):
    s = Span(name, kind); s.inputs = inputs; s.start = time.perf_counter()
    token = _current.set(s)
    try:
        yield s
    except Exception as e:
        s.status = "error"; s.error = repr(e)
        raise
    finally:
        s.end = time.perf_counter()
        _current.reset(token)

def log(**fields):
    # a structured log line, attached to the current span
    s = _current.get()
    line = dict(ts=round(time.time(), 3), span=s.name if s else None, **fields)
    if s: s.metadata.setdefault("logs", []).append(line)
    return line

def render(span_, indent=0):
    d = f"{span_.duration_ms:6.1f}ms" if span_.duration_ms is not None else "   --   "
    mark = "  " if span_.status == "ok" else "!!"
    print(f"{'  '*indent}{mark} {span_.name:22s} {d}  "
          f"in={_short(span_.inputs)}  out={_short(span_.outputs)}")
    for c in span_.children:
        render(c, indent + 1)

def _short(d, n=60):
    s = json.dumps(d, default=str)
    return s if len(s) <= n else s[:n] + "…}"

with span("demo_trace") as root:
    with span("step_a", x=1) as a:
        time.sleep(0.005); a.outputs = {"y": 2}
    with span("step_b") as b:
        with span("nested") as nn:
            time.sleep(0.003); nn.outputs = {"ok": True}
        b.outputs = {"done": True}
render(root)

   demo_trace               11.5ms  in={}  out={}
     step_a                    6.9ms  in={"x": 1}  out={"y": 2}
     step_b                    4.5ms  in={}  out={"done": true}
       nested                    4.5ms  in={}  out={"ok": true}


## 2 — Instrument a RAG pipeline (12 min)

Wrap each stage in a `span`. Nothing about the pipeline logic changes — you're adding a
recording layer.

In [3]:
KB = {
 "pto": "Full-time staff accrue 15 vacation days in year one; up to 10 unused days carry over.",
 "expenses": "Receipts are required for expenses of $25 or more. Meal cap is $75 domestic, $100 international.",
 "travel": "Book travel 14 days ahead. The standard hotel cap is $250 per night. A higher cap of $350 applies in New York, San Francisco and London.",
 "security": "Laptops auto-lock after 10 minutes idle. Production access is granted for 90 days.",
}
KEYS = list(KB); KVEC = E(list(KB.values()))
PRICE_IN, PRICE_OUT = 1/1e6, 5/1e6

def rewrite_query(q):
    with span("rewrite_query", kind="transform", query=q) as s:
        rq = q.strip().rstrip("?").lower()
        rq = rq.replace("holiday", "vacation").replace("pto", "vacation")
        s.outputs = {"rewritten": rq}
        return rq

def retrieve(q, k=3):
    with span("retrieve", kind="retriever", query=q, k=k) as s:
        sims = KVEC @ E(q)[0]
        idx = np.argsort(-sims)[:k]
        hits = [dict(doc=KEYS[i], score=round(float(sims[i]), 3)) for i in idx]
        log(n_hits=len(hits), top_score=hits[0]["score"], top_doc=hits[0]["doc"])
        s.outputs = {"hits": hits}
        return [(h["doc"], KB[h["doc"]], h["score"]) for h in hits]

def build_prompt(q, chunks):
    with span("build_prompt", kind="transform") as s:
        ctx = "\n".join(f"- {c[1]}" for c in chunks)
        prompt = f"Answer from context only.\nContext:\n{ctx}\nQ: {q}\nA:"
        s.outputs = {"prompt_sha": hashlib.sha1(prompt.encode()).hexdigest()[:10],
                     "prompt_tokens": len(prompt) // 4}
        return prompt

def generate(prompt, q, chunks, fail=False):
    with span("generate", kind="llm", model="mock-llm") as s:
        s.inputs = {"prompt_tokens": len(prompt) // 4}
        time.sleep(0.004)
        if fail or not chunks:
            ans = "I don't have information on that."
        else:
            # a deliberately naive extractor: first sentence sharing a content word with the
            # query. Fast, and it will sometimes grab a near-miss sentence (see the §4 bug).
            import re as _re
            STOP = {"the","and","are","for","get","what","how","does","you","your","with"}
            words = {w for w in _re.findall(r"[a-z]{3,}", q.lower()) if w not in STOP}
            ans = "I don't have information on that."
            for sent in chunks[0][1].split(". "):
                if any(_re.search(rf"\b{w}", sent.lower()) for w in words):
                    ans = sent.strip(". ") + "."; break
        tok_in, tok_out = len(prompt)//4, len(ans)//4
        cost = tok_in*PRICE_IN + tok_out*PRICE_OUT
        s.outputs = {"answer": ans, "tok_in": tok_in, "tok_out": tok_out, "cost_usd": round(cost, 6)}
        log(cost_usd=round(cost, 6), tok_out=tok_out)
        return ans

def rag(query, gate=0.25, fail_gen=False):
    with span("rag_request", kind="chain", user_query=query) as root:
        root.metadata["trace_id"] = root.id
        rq = rewrite_query(query)
        chunks = retrieve(rq)
        with span("relevance_gate", kind="guard") as g:
            gated = chunks[0][2] < gate
            g.outputs = {"gated": gated, "top_score": chunks[0][2], "threshold": gate}
        if gated:
            root.outputs = {"answer": "I don't have information on that.", "gated": True}
            return root.outputs["answer"], root
        prompt = build_prompt(rq, chunks)
        ans = generate(prompt, rq, chunks, fail=fail_gen)
        root.outputs = {"answer": ans}
        return ans, root

ans, trace = rag("how many holiday days do I get in year one?")
print("ANSWER:", ans, "\n")
render(trace)

ANSWER: Full-time staff accrue 15 vacation days in year one; up to 10 unused days carry over. 

   rag_request              20.1ms  in={"user_query": "how many holiday days do I get in year one?"…}  out={"answer": "Full-time staff accrue 15 vacation days in year …}
     rewrite_query             0.0ms  in={"query": "how many holiday days do I get in year one?"}  out={"rewritten": "how many vacation days do i get in year one"}
     retrieve                 12.6ms  in={"query": "how many vacation days do i get in year one", "k"…}  out={"hits": [{"doc": "pto", "score": 0.681}, {"doc": "travel", …}
     relevance_gate            0.0ms  in={}  out={"gated": false, "top_score": 0.681, "threshold": 0.25}
     build_prompt              1.3ms  in={}  out={"prompt_sha": "ee994c92d4", "prompt_tokens": 99}
     generate                  6.2ms  in={"prompt_tokens": 99}  out={"answer": "Full-time staff accrue 15 vacation days in year …}


## 3 — What to capture at each stage (10 min)

| Stage | Capture | Don't capture (or redact) |
| ----- | ------- | ------------------------- |
| request | trace id, user id (hashed), timestamp, app version, model + params | raw PII in the query |
| query rewrite | original + rewritten query | — |
| retrieve | query, k, retrieved **ids + scores**, index/version | full chunk text (store ids; fetch on demand) |
| rerank | before/after order, scores | — |
| guard/gate | the decision + the threshold + the value it compared | — |
| build prompt | **hash + token count** of the final prompt (full text only when sampled) | the full prompt on every request (cost, PII) |
| generate | model, params, `stop_reason`, `usage`, latency, **cost**, raw response | — |
| tool call | tool name, args, result, latency, error | secrets in args |
| parse/validate | success/failure, the schema, what failed | — |
| output | final answer, any online eval scores, guardrail hits | — |

Rule: capture enough to **reconstruct the decision** at every branch. Scores and ids, not
walls of text. Attach a stable `trace_id` and surface it in the UI / error messages so a
user's "it gave me a wrong answer" maps to one trace.

In [4]:
# export the trace flat (what you'd ship to a backend / store in a table)
def flatten(span_, out=None):
    out = [] if out is None else out
    out.append(dict(trace_id=_root(span_).id, span_id=span_.id,
                    parent=span_.parent.id if span_.parent else None,
                    name=span_.name, kind=span_.kind, status=span_.status,
                    duration_ms=round(span_.duration_ms, 2) if span_.duration_ms else None,
                    inputs=span_.inputs, outputs=span_.outputs, metadata=span_.metadata))
    for c in span_.children: flatten(c, out)
    return out

def _root(s):
    while s.parent: s = s.parent
    return s

print(json.dumps(flatten(trace)[:3], indent=1, default=str))
print(f"\ntotal spans: {len(flatten(trace))}")

[
 {
  "trace_id": "465a1331",
  "span_id": "465a1331",
  "parent": null,
  "name": "rag_request",
  "kind": "chain",
  "status": "ok",
  "duration_ms": 20.08,
  "inputs": {
   "user_query": "how many holiday days do I get in year one?"
  },
  "outputs": {
   "answer": "Full-time staff accrue 15 vacation days in year one; up to 10 unused days carry over."
  },
  "metadata": {
   "trace_id": "465a1331"
  }
 },
 {
  "trace_id": "465a1331",
  "span_id": "cccec468",
  "parent": "465a1331",
  "name": "rewrite_query",
  "kind": "transform",
  "status": "ok",
  "duration_ms": 0.01,
  "inputs": {
   "query": "how many holiday days do I get in year one?"
  },
  "outputs": {
   "rewritten": "how many vacation days do i get in year one"
  },
  "metadata": {}
 },
 {
  "trace_id": "465a1331",
  "span_id": "885c13af",
  "parent": "465a1331",
  "name": "retrieve",
  "kind": "retriever",
  "status": "ok",
  "duration_ms": 12.55,
  "inputs": {
   "query": "how many vacation days do i get in year one",


## 4 — Debugging from a trace (10 min)

A user reports: *"I asked about the London hotel cap and got told $250."* Find the trace, walk
the spans.

In [5]:
ans, trace = rag("what's the hotel cap for London")
print("ANSWER:", ans, "\n")
render(trace)

# walk the span tree to localise the fault
def find(span_, name):
    if span_.name == name: return span_
    for c in span_.children:
        r = find(c, name)
        if r: return r
    return None

ret = find(trace, "retrieve")
gen = find(trace, "generate")
print("\n-- diagnosis --")
print("retrieved:", [(h['doc'], h['score']) for h in ret.outputs['hits']])
print("top doc contains 'London'? ", "London" in KB[ret.outputs['hits'][0]['doc']])
print("generate saw context from:", ret.outputs['hits'][0]['doc'])
print("\n-> retrieval put 'travel' on top (good, and the $350/London sentence IS in the "
      "chunk), but the naive generator grabbed the first sentence mentioning 'hotel cap' -- "
      "the $250 one -- and stopped. The bug is in GENERATE, not retrieval. Fix: a better "
      "prompt / model, or a sentence-level reranker before extraction.")

ANSWER: The standard hotel cap is $250 per night. 

   rag_request              16.0ms  in={"user_query": "what's the hotel cap for London"}  out={"answer": "The standard hotel cap is $250 per night."}
     rewrite_query             0.0ms  in={"query": "what's the hotel cap for London"}  out={"rewritten": "what's the hotel cap for london"}
     retrieve                  9.8ms  in={"query": "what's the hotel cap for london", "k": 3}  out={"hits": [{"doc": "travel", "score": 0.643}, {"doc": "expens…}
     relevance_gate            0.0ms  in={}  out={"gated": false, "top_score": 0.643, "threshold": 0.25}
     build_prompt              0.0ms  in={}  out={"prompt_sha": "e7e307e695", "prompt_tokens": 99}
     generate                  6.1ms  in={"prompt_tokens": 99}  out={"answer": "The standard hotel cap is $250 per night.", "tok…}

-- diagnosis --
retrieved: [('travel', 0.643), ('expenses', 0.226), ('pto', 0.111)]
top doc contains 'London'?  True
generate saw context from: travel

-> retri

The trace turns "the answer is wrong" into "**span X** produced the wrong output given a
**correct input**" — which tells you exactly which component to fix, and gives you the input
to add as a Day 26 test case.

### The workflow

1. User report / thumbs-down / alert → get the `trace_id`.
2. Open the trace, read top to bottom. Find the first span whose **output is wrong given a
   correct input**.
3. That span is the fault. Fix it.
4. Add the failing input as a test case (Day 26) so the fix is protected.
5. If many traces show the same span failing → a metric/alert on that span's quality.

## 5 — Redaction, sampling, cost; the real tools (7 min)

- **PII:** redact emails/phones/names from logged queries and prompts (regex + a NER pass);
  log a hash of the user id, never the raw one.
- **Sampling:** store full prompt/response text for 1–10% of traces (or 100% of errors and
  thumbs-downs); store ids + scores + hashes for the rest. Full-text logging of every request
  is a storage and privacy liability.
- **Retention:** short TTL (7–30 days) for full traces; longer for aggregates.
- **Cost of observability:** the tracing overhead is tiny; the *storage* isn't. Budget it.

### The tools (they all do the same core thing)

| Tool | Notes |
| ---- | ----- |
| **LangSmith** | LangChain-native; `@traceable` decorator or automatic if you use LCEL; datasets + eval integration |
| **Langfuse** | OSS, self-hostable; SDK `@observe` decorator; prompt management + evals |
| **Arize Phoenix** | OSS; OpenTelemetry-based; strong on retrieval/embedding debugging |
| **OpenLLMetry / OpenTelemetry GenAI** | vendor-neutral spans; export anywhere (Datadog, Grafana, …) |
| **Braintrust / Weights & Biases Weave** | eval-forward, experiment tracking |

They give you: a hosted span store + UI, automatic instrumentation for common libraries, and a
link from a trace to your eval datasets. What you built here is the data model underneath all
of them — a `@traceable`/`@observe` decorator is exactly the `with span(...)` context manager.

In [6]:
# the decorator form (what LangSmith @traceable / Langfuse @observe give you)
def traceable(kind="op"):
    def deco(fn):
        def wrapped(*a, **kw):
            with span(fn.__name__, kind=kind, args=_short({"a": a, "kw": kw})) as s:
                r = fn(*a, **kw)
                s.outputs = {"result": _short({"r": r})}
                return r
        return wrapped
    return deco

@traceable(kind="retriever")
def my_retrieve(q):
    return retrieve(rewrite_query(q))

with span("decorated_demo") as root:
    my_retrieve("holiday allowance")
render(root)

   decorated_demo           21.2ms  in={}  out={}
     my_retrieve              21.1ms  in={"args": "{\"a\": [\"holiday allowance\"], \"kw\": {}}"}  out={"result": "{\"r\": [[\"pto\", \"Full-time staff accrue 15 v…}
       rewrite_query             0.0ms  in={"query": "holiday allowance"}  out={"rewritten": "vacation allowance"}
       retrieve                 21.1ms  in={"query": "vacation allowance", "k": 3}  out={"hits": [{"doc": "pto", "score": 0.554}, {"doc": "travel", …}


## 6 — Exercises

1. **Add token/cost rollup.** Walk the trace and sum `tok_in`, `tok_out`, `cost_usd` across
   all `llm` spans; attach the totals to the root span's metadata. Print a one-line cost
   summary per request.
2. **Slowest-span report.** Over 20 simulated requests, aggregate mean and p95 `duration_ms`
   per span name. Which stage is the latency bottleneck?
3. **Error propagation.** Make `generate` raise on an empty prompt. Show the exception marks
   the `generate` span `status="error"`, propagates to `rag_request`, and `render` shows `!!`
   on both — while sibling spans stay `ok`.
4. **PII redaction.** Write `redact(text)` that masks emails, phone numbers, and 16-digit
   card numbers. Apply it in `rewrite_query`'s span inputs. Verify a query with an email is
   stored redacted.
5. **Sampled full-text.** Add a `sample_rate` to `rag`; when a coin flip says "sample",
   store the full prompt text in the `build_prompt` span, otherwise only the hash. Over 100
   requests, confirm ~`sample_rate` have full text.
6. **Trace → test case.** From the London-hotel trace, produce a Day-26-style test dict
   (`id`, `type`, `q`, `fact="$350"`, `source="travel"`) automatically from the span data.

In [7]:
# ---- Solution 1 ----
def cost_rollup(root):
    total = dict(tok_in=0, tok_out=0, cost_usd=0.0)
    def walk(s):
        if s.kind == "llm":
            for kk in total: total[kk] += s.outputs.get(kk, 0)
        for c in s.children: walk(c)
    walk(root)
    root.metadata["cost_rollup"] = {k: round(v, 6) for k, v in total.items()}
    return total

_, tr = rag("how far ahead should I book travel")
r = cost_rollup(tr)
print(f"S1: request cost ${r['cost_usd']:.6f}  ({r['tok_in']} in / {r['tok_out']} out)")

S1: request cost $0.000130  (100 in / 6 out)


In [8]:
# ---- Solution 3 ----
def generate_strict(prompt, q, chunks):
    with span("generate", kind="llm") as s:
        if not prompt.strip(): raise ValueError("empty prompt")
        s.outputs = {"answer": "ok"}; return "ok"

with span("err_demo", kind="chain") as root:
    try:
        with span("sibling_ok") as sib: sib.outputs = {"fine": True}
        generate_strict("", "q", [])
    except ValueError:
        root.status = "error"
render(root)
print("S3: 'generate' and 'err_demo' show !! ; 'sibling_ok' stays ok.")

!! err_demo                  0.0ms  in={}  out={}
     sibling_ok                0.0ms  in={}  out={"fine": true}
  !! generate                  0.0ms  in={}  out={}
S3: 'generate' and 'err_demo' show !! ; 'sibling_ok' stays ok.


In [9]:
# ---- Solution 6 ----
def trace_to_testcase(root):
    ret = None
    def walk(s):
        nonlocal ret
        if s.name == "retrieve": ret = s
        for c in s.children: walk(c)
    walk(root)
    q = root.inputs["user_query"]
    return dict(id="prod_" + root.id, type="paraphrase" if "?" not in q else "single_fact",
                q=q, fact=None, source=ret.outputs["hits"][0]["doc"] if ret else None)

_, tr = rag("what's the hotel cap for London")
print("S6:", json.dumps(trace_to_testcase(tr), indent=1))
print("    (a human fills in the correct `fact` -- '$350' -- then it joins the eval set)")

S6: {
 "id": "prod_673a28bc",
 "type": "paraphrase",
 "q": "what's the hotel cap for London",
 "fact": null,
 "source": "travel"
}
    (a human fills in the correct `fact` -- '$350' -- then it joins the eval set)


### Solutions 2, 4, 5 (sketch)

**S2:** `by_name = defaultdict(list); for req in range(20): _, t = rag(q); for s in
flatten_objs(t): by_name[s.name].append(s.duration_ms)`. Then mean + `np.percentile(v, 95)`
per name. `generate` (the LLM call) is almost always the bottleneck; `retrieve` second.

**S4:** `re.sub(r"[\w.]+@[\w.]+", "[EMAIL]", text)`, `re.sub(r"\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b",
"[PHONE]", ...)`, `re.sub(r"\b\d{4}[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b", "[CARD]", ...)`. Apply
before assigning `s.inputs`. The point: redaction happens at the *instrumentation* layer so no
raw PII ever reaches the trace store.

**S5:** `store_full = random.random() < sample_rate`; in `build_prompt`,
`s.outputs["prompt"] = prompt if store_full else None`. Over 100 runs, `~sample_rate` spans
have non-None `prompt`. Always store full text for `status == "error"` regardless of the coin.

## Self-check quiz

1. Name the three pillars of observability and the question each answers.
2. Why is the *trace* the most important pillar for an LLM app specifically?
3. What is a span, and what makes a set of spans a trace?
4. At the retrieve stage, what should you log and what should you *not* log?
5. Walk the debugging workflow from a user's thumbs-down to a code fix.
6. Why sample full prompt/response text instead of storing it for every request?
7. What is a `@traceable` / `@observe` decorator, in terms of what you built today?

### Answer key

1. Logs (what happened at this moment — structured events); traces (the full path of one
   request — a tree of spans); metrics (aggregate health over time — numbers).
2. One request fans out into retrieval, reranking, tool calls, and generation; a wrong final
   answer can originate in any of them, and only the trace shows each stage's input and
   output so you can localise the fault.
3. A span is one timed operation with a name, start/end, inputs, outputs, metadata, status,
   and a parent. A trace is a root span plus all its descendants — one request.
4. Log: the query, k, retrieved ids + scores, index version. Don't log: the full chunk text
   on every request (store ids, fetch on demand) or raw PII.
5. Get the trace_id from the report/alert → open the trace, read top to bottom → find the
   first span whose output is wrong given a correct input → that span is the fault, fix it →
   add the failing input as a Day-26 test case → if it recurs, add a metric on that span.
6. Full-text logging of every request is a storage cost and a privacy/PII liability; sampling
   1–10% (plus 100% of errors/thumbs-downs) keeps debuggability while bounding both.
7. It's the `with span(...)` context manager wrapped around a function — it records the
   function's args as span inputs and its return value as span outputs, automatically,
   wherever the function is called.

## Where this goes next

Week 9 done — you can eval offline and observe in production. **Week 10 — Deployment**: AWS
Bedrock, model serving and cost at scale, and shipping the Week 6 RAG pipeline as an API
endpoint (Day 28).